# ASL MNIST — trening eksperymentów

Ten notebook trenuje modele z konfiguracji Fiddle. Wyniki i wagi modeli są zapisywane do `artifacts/`, żeby osobny notebook mógł wykonać analizę wyników.

In [1]:
import sys
import traceback
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import fiddle as fdl
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display

from configs.constants import (
    EXPERIMENT_HISTORY_CSV,
    EXPERIMENT_RESULTS_CSV,
    MODEL_CHECKPOINTS_DIR,
)
from configs.experiments import all_experiments
from src.helpers import (
    build_model,
    build_optimizer,
    history_to_rows,
    make_data_loaders,
    plot_experiment_comparison,
    plot_training_history,
    resolve_device,
    save_model_checkpoint,
    save_training_tables,
    set_seed,
    train_model,
)

## Lista eksperymentów

Eksperymenty pochodzą z `configs/experiments/registry.py`, który scala sweepe MLP, LeNet i CNN.

In [2]:
experiments = [fdl.build(config) for config in all_experiments()]

experiment_plan_df = pd.DataFrame(
    [
        {
            "name": experiment.name,
            "model": experiment.model.name,
            "optimizer": experiment.optimizer.name,
            "learning_rate": experiment.optimizer.learning_rate,
            "momentum": experiment.optimizer.momentum,
            "epochs": experiment.epochs,
            "batch_size": experiment.dataset.batch_size,
        }
        for experiment in experiments
    ]
)

experiment_plan_df

,name,model,optimizer,learning_rate,momentum,epochs,batch_size
0,mlp_adam_lr_1e_3,mlp,adam,0.0010,0.0,4,64
1,mlp_adam_lr_3e_4,mlp,adam,0.0003,0.0,4,64
2,mlp_sgd_lr_1e_2,mlp,sgd,0.0100,0.0,4,64
3,mlp_sgd_momentum_lr_1e_2,mlp,sgd,0.0100,0.9,4,64
4,lenet_adam_lr_1e_3,lenet,adam,0.0010,0.0,4,64
5,lenet_adam_lr_3e_4,lenet,adam,0.0003,0.0,4,64
6,lenet_sgd_lr_1e_2,lenet,sgd,0.0100,0.0,4,64
7,lenet_sgd_momentum_lr_1e_2,lenet,sgd,0.0100,0.9,4,64
8,cnn_adam_lr_1e_3,cnn,adam,0.0010,0.0,4,64
9,cnn_adam_lr_3e_4,cnn,adam,0.0003,0.0,4,64


In [3]:
pd.DataFrame(
    [{"model": experiment.model.name} for experiment in experiments]
).value_counts("model").rename("experiments").reset_index()

,model,experiments
0,cnn,4
1,lenet,4
2,mlp,4


## Testy konfiguracji

Przed treningiem sprawdzamy, czy rejestr eksperymentów jest spójny: nazwy są unikalne, każdy typ modelu ma po 4 warianty, a model i optimizer dają się zbudować dla każdej konfiguracji.

In [4]:
expected_models = {"mlp", "lenet", "cnn"}
model_counts = experiment_plan_df.value_counts("model").to_dict()

assert set(experiment_plan_df["model"]) == expected_models
assert experiment_plan_df["name"].is_unique
assert all(model_counts[model] == 4 for model in expected_models)
assert len(experiments) == 12

smoke_test_rows = []
for experiment in experiments:
    model = build_model(experiment.model)
    optimizer = build_optimizer(experiment.optimizer, model.parameters())
    total_params = sum(p.numel() for p in model.parameters())

    if experiment.model.name == "mlp":
        dummy_input = torch.randn(2, experiment.model.input_size)
        flatten = True
    else:
        dummy_input = torch.randn(2, 1, 28, 28)
        flatten = False

    output = model(dummy_input)
    assert tuple(output.shape) == (2, experiment.model.num_classes)

    smoke_test_rows.append(
        {
            "name": experiment.name,
            "model": experiment.model.name,
            "optimizer_class": optimizer.__class__.__name__,
            "parameters": total_params,
            "output_shape": tuple(output.shape),
            "flatten_input": flatten,
        }
    )

pd.DataFrame(smoke_test_rows)

,name,model,optimizer_class,parameters,output_shape,flatten_input
0,mlp_adam_lr_1e_3,mlp,Adam,207128,"(2, 24)",True
1,mlp_adam_lr_3e_4,mlp,Adam,207128,"(2, 24)",True
2,mlp_sgd_lr_1e_2,mlp,SGD,207128,"(2, 24)",True
3,mlp_sgd_momentum_lr_1e_2,mlp,SGD,207128,"(2, 24)",True
4,lenet_adam_lr_1e_3,lenet,Adam,45616,"(2, 24)",False
5,lenet_adam_lr_3e_4,lenet,Adam,45616,"(2, 24)",False
6,lenet_sgd_lr_1e_2,lenet,SGD,45616,"(2, 24)",False
7,lenet_sgd_momentum_lr_1e_2,lenet,SGD,45616,"(2, 24)",False
8,cnn_adam_lr_1e_3,cnn,Adam,394456,"(2, 24)",False
9,cnn_adam_lr_3e_4,cnn,Adam,394456,"(2, 24)",False


## Trening

Każdy eksperyment jest trenowany po 4 epoki. Dla MLP obrazy są spłaszczane do wektora 784, a dla LeNet i CNN zostają w kształcie `1 x 28 x 28`.

In [5]:
experiment_histories = {}
experiment_results = []
experiment_history_rows = []
failed_experiments = []

for experiment in experiments:
    print("=" * 80)
    print(f"Experiment: {experiment.name}")
    print(
        f"Model: {experiment.model.name}, "
        f"optimizer: {experiment.optimizer.name}, "
        f"lr={experiment.optimizer.learning_rate}, "
        f"momentum={experiment.optimizer.momentum}, "
        f"weight_decay={experiment.optimizer.weight_decay}"
    )

    try:
        set_seed(experiment.seed)
        experiment_device = resolve_device(experiment.device)
        train_loader, test_loader = make_data_loaders(
            experiment.dataset,
            root_dir=PROJECT_ROOT,
        )

        model = build_model(experiment.model).to(experiment_device)
        criterion = nn.CrossEntropyLoss()
        optimizer = build_optimizer(experiment.optimizer, model.parameters())

        total_params = sum(p.numel() for p in model.parameters())
        flatten = experiment.model.name == "mlp"
        print(f"Device: {experiment_device}")
        print(f"Liczba parametrów: {total_params:,}")

        history = train_model(
            model,
            train_loader,
            test_loader,
            criterion,
            optimizer,
            experiment_device,
            epochs=experiment.epochs,
            flatten=flatten,
        )

        best_val_acc = max(history["val_acc"])
        best_epoch = history["val_acc"].index(best_val_acc) + 1
        checkpoint_path = save_model_checkpoint(
            model,
            experiment,
            output_dir=PROJECT_ROOT / MODEL_CHECKPOINTS_DIR,
            history=history,
            extra_metadata={
                "best_val_acc": best_val_acc,
                "best_epoch": best_epoch,
                "total_params": total_params,
            },
        )

        experiment_histories[experiment.name] = history
        experiment_history_rows.extend(history_to_rows(experiment, history))
        experiment_results.append(
            {
                "name": experiment.name,
                "model": experiment.model.name,
                "optimizer": experiment.optimizer.name,
                "learning_rate": experiment.optimizer.learning_rate,
                "momentum": experiment.optimizer.momentum,
                "weight_decay": experiment.optimizer.weight_decay,
                "epochs": experiment.epochs,
                "total_params": total_params,
                "best_val_acc": best_val_acc,
                "best_epoch": best_epoch,
                "checkpoint_path": str(checkpoint_path),
            }
        )

        display(plot_training_history(history, title=f"{experiment.name} — {experiment.epochs} epoki"))
        print(f"Najlepszy val acc: {best_val_acc:.4f} (epoka {best_epoch})")
        print(f"Checkpoint: {checkpoint_path}")

    except Exception as error:
        failed_experiments.append(
            {
                "name": experiment.name,
                "model": experiment.model.name,
                "error": repr(error),
            }
        )
        print(f"FAILED: {experiment.name}")
        print(traceback.format_exc())

if failed_experiments:
    display(pd.DataFrame(failed_experiments))


Experiment: mlp_adam_lr_1e_3
Model: mlp, optimizer: adam, lr=0.001, momentum=0.0, weight_decay=0.0
Device: mps
Liczba parametrów: 207,128
Epoch   1/4 | Train loss: 2.2415  acc: 0.3498 | Val loss: 1.6717  acc: 0.5208
Epoch   2/4 | Train loss: 1.3497  acc: 0.5987 | Val loss: 1.4211  acc: 0.5717
Epoch   3/4 | Train loss: 1.0293  acc: 0.6930 | Val loss: 1.2440  acc: 0.6122
Epoch   4/4 | Train loss: 0.8403  acc: 0.7531 | Val loss: 1.1165  acc: 0.6570


Najlepszy val acc: 0.6570 (epoka 4)
Checkpoint: /Users/piotrpijanowski/Documents/Studia/Semestr 6/neural_networks/project/artifacts/models/mlp_adam_lr_1e_3.pt
Experiment: mlp_adam_lr_3e_4
Model: mlp, optimizer: adam, lr=0.0003, momentum=0.0, weight_decay=0.0
Device: mps
Liczba parametrów: 207,128
Epoch   1/4 | Train loss: 2.5662  acc: 0.3021 | Val loss: 2.0417  acc: 0.4541
Epoch   2/4 | Train loss: 1.7041  acc: 0.5366 | Val loss: 1.6555  acc: 0.5279
Epoch   3/4 | Train loss: 1.3407  acc: 0.6333 | Val loss: 1.4282  acc: 0.5545
Epoch   4/4 | Train loss: 1.1311  acc: 0.6877 | Val loss: 1.3060  acc: 0.5952


Najlepszy val acc: 0.5952 (epoka 4)
Checkpoint: /Users/piotrpijanowski/Documents/Studia/Semestr 6/neural_networks/project/artifacts/models/mlp_adam_lr_3e_4.pt
Experiment: mlp_sgd_lr_1e_2
Model: mlp, optimizer: sgd, lr=0.01, momentum=0.0, weight_decay=0.0
Device: mps
Liczba parametrów: 207,128
Epoch   1/4 | Train loss: 3.1003  acc: 0.1231 | Val loss: 3.0394  acc: 0.1820
Epoch   2/4 | Train loss: 2.8550  acc: 0.2874 | Val loss: 2.7185  acc: 0.3008
Epoch   3/4 | Train loss: 2.4979  acc: 0.3780 | Val loss: 2.3641  acc: 0.3935
Epoch   4/4 | Train loss: 2.1718  acc: 0.4435 | Val loss: 2.1021  acc: 0.4842


Najlepszy val acc: 0.4842 (epoka 4)
Checkpoint: /Users/piotrpijanowski/Documents/Studia/Semestr 6/neural_networks/project/artifacts/models/mlp_sgd_lr_1e_2.pt
Experiment: mlp_sgd_momentum_lr_1e_2
Model: mlp, optimizer: sgd, lr=0.01, momentum=0.9, weight_decay=0.0
Device: mps
Liczba parametrów: 207,128
Epoch   1/4 | Train loss: 2.2360  acc: 0.3470 | Val loss: 1.6128  acc: 0.5266
Epoch   2/4 | Train loss: 1.1837  acc: 0.6349 | Val loss: 1.2883  acc: 0.5936
Epoch   3/4 | Train loss: 0.8122  acc: 0.7547 | Val loss: 1.1255  acc: 0.6393
Epoch   4/4 | Train loss: 0.6099  acc: 0.8152 | Val loss: 1.0677  acc: 0.6481


Najlepszy val acc: 0.6481 (epoka 4)
Checkpoint: /Users/piotrpijanowski/Documents/Studia/Semestr 6/neural_networks/project/artifacts/models/mlp_sgd_momentum_lr_1e_2.pt
Experiment: lenet_adam_lr_1e_3
Model: lenet, optimizer: adam, lr=0.001, momentum=0.0, weight_decay=0.0
Device: mps
Liczba parametrów: 45,616
Epoch   1/4 | Train loss: 1.8636  acc: 0.4114 | Val loss: 1.1473  acc: 0.6117
Epoch   2/4 | Train loss: 0.6042  acc: 0.8058 | Val loss: 0.6058  acc: 0.7893
Epoch   3/4 | Train loss: 0.2519  acc: 0.9233 | Val loss: 0.5181  acc: 0.8250
Epoch   4/4 | Train loss: 0.1019  acc: 0.9738 | Val loss: 0.5550  acc: 0.8560


...

Najlepszy val acc: 0.8666 (epoka 4)
Checkpoint: /Users/piotrpijanowski/Documents/Studia/Semestr 6/neural_networks/project/artifacts/models/lenet_sgd_momentum_lr_1e_2.pt
Experiment: cnn_adam_lr_1e_3
Model: cnn, optimizer: adam, lr=0.001, momentum=0.0, weight_decay=0.0
Device: mps
Liczba parametrów: 394,456
Epoch   1/4 | Train loss: 0.3837  acc: 0.8870 | Val loss: 0.1492  acc: 0.9532
Epoch   2/4 | Train loss: 0.0241  acc: 0.9940 | Val loss: 0.1063  acc: 0.9590
Epoch   3/4 | Train loss: 0.0236  acc: 0.9932 | Val loss: 0.0926  acc: 0.9622
Epoch   4/4 | Train loss: 0.0098  acc: 0.9972 | Val loss: 0.0826  acc: 0.9679


Najlepszy val acc: 0.9679 (epoka 4)
Checkpoint: /Users/piotrpijanowski/Documents/Studia/Semestr 6/neural_networks/project/artifacts/models/cnn_adam_lr_1e_3.pt
Experiment: cnn_adam_lr_3e_4
Model: cnn, optimizer: adam, lr=0.0003, momentum=0.0, weight_decay=0.0
Device: mps
Liczba parametrów: 394,456
Epoch   1/4 | Train loss: 0.6770  acc: 0.8212 | Val loss: 0.2007  acc: 0.9402
Epoch   2/4 | Train loss: 0.0431  acc: 0.9946 | Val loss: 0.1211  acc: 0.9689
Epoch   3/4 | Train loss: 0.0154  acc: 0.9986 | Val loss: 0.1005  acc: 0.9608
Epoch   4/4 | Train loss: 0.0115  acc: 0.9985 | Val loss: 0.1268  acc: 0.9582


Najlepszy val acc: 0.9689 (epoka 2)
Checkpoint: /Users/piotrpijanowski/Documents/Studia/Semestr 6/neural_networks/project/artifacts/models/cnn_adam_lr_3e_4.pt
Experiment: cnn_sgd_lr_1e_2
Model: cnn, optimizer: sgd, lr=0.01, momentum=0.0, weight_decay=0.0
Device: mps
Liczba parametrów: 394,456
Epoch   1/4 | Train loss: 1.5888  acc: 0.5758 | Val loss: 0.5929  acc: 0.8799
Epoch   2/4 | Train loss: 0.3256  acc: 0.9286 | Val loss: 0.2696  acc: 0.9331
Epoch   3/4 | Train loss: 0.1346  acc: 0.9796 | Val loss: 0.1924  acc: 0.9499
Epoch   4/4 | Train loss: 0.0736  acc: 0.9909 | Val loss: 0.1475  acc: 0.9591


Najlepszy val acc: 0.9591 (epoka 4)
Checkpoint: /Users/piotrpijanowski/Documents/Studia/Semestr 6/neural_networks/project/artifacts/models/cnn_sgd_lr_1e_2.pt
Experiment: cnn_sgd_momentum_lr_1e_2
Model: cnn, optimizer: sgd, lr=0.01, momentum=0.9, weight_decay=0.0
Device: mps
Liczba parametrów: 394,456
Epoch   1/4 | Train loss: 0.5073  acc: 0.8445 | Val loss: 0.1680  acc: 0.9439
Epoch   2/4 | Train loss: 0.0244  acc: 0.9942 | Val loss: 0.0567  acc: 0.9780
Epoch   3/4 | Train loss: 0.0116  acc: 0.9976 | Val loss: 0.0433  acc: 0.9834
Epoch   4/4 | Train loss: 0.0097  acc: 0.9976 | Val loss: 0.0415  acc: 0.9842


Najlepszy val acc: 0.9842 (epoka 4)
Checkpoint: /Users/piotrpijanowski/Documents/Studia/Semestr 6/neural_networks/project/artifacts/models/cnn_sgd_momentum_lr_1e_2.pt


## Zapis wyników

Po wykonaniu tej komórki notebook `03_results_analysis.ipynb` może wczytać tabele wyników i historii treningu.

In [6]:
if not experiment_results:
    raise RuntimeError(
        "Brak wyników do zapisania. Uruchom komórkę treningu i sprawdź tabelę failed_experiments."
    )

experiment_results_df = pd.DataFrame(experiment_results).sort_values(
    "best_val_acc",
    ascending=False,
).reset_index(drop=True)

save_training_tables(
    experiment_results_df,
    experiment_history_rows,
    results_csv=PROJECT_ROOT / EXPERIMENT_RESULTS_CSV,
    history_csv=PROJECT_ROOT / EXPERIMENT_HISTORY_CSV,
)

if failed_experiments:
    print("Nie wszystkie eksperymenty zakończyły się poprawnie:")
    display(pd.DataFrame(failed_experiments))

display(plot_experiment_comparison(
    experiment_results_df,
    metric="best_val_acc",
    title="Porównanie eksperymentów",
))

experiment_results_df


,name,model,optimizer,learning_rate,momentum,weight_decay,epochs,total_params,best_val_acc,best_epoch,checkpoint_path
0,cnn_sgd_momentum_lr_1e_2,cnn,sgd,0.0100,0.9,0.0,4,394456,0.984244,4,/Users/piotrpijanowski/Documents/Studia/Semest...
1,cnn_adam_lr_3e_4,cnn,adam,0.0003,0.0,0.0,4,394456,0.968907,2,/Users/piotrpijanowski/Documents/Studia/Semest...
2,cnn_adam_lr_1e_3,cnn,adam,0.0010,0.0,0.0,4,394456,0.967931,4,/Users/piotrpijanowski/Documents/Studia/Semest...
3,cnn_sgd_lr_1e_2,cnn,sgd,0.0100,0.0,0.0,4,394456,0.959147,4,/Users/piotrpijanowski/Documents/Studia/Semest...
4,lenet_sgd_momentum_lr_1e_2,lenet,sgd,0.0100,0.9,0.0,4,45616,0.866564,4,/Users/piotrpijanowski/Documents/Studia/Semest...
5,lenet_adam_lr_1e_3,lenet,adam,0.0010,0.0,0.0,4,45616,0.855968,4,/Users/piotrpijanowski/Documents/Studia/Semest...
6,lenet_adam_lr_3e_4,lenet,adam,0.0003,0.0,0.0,4,45616,0.702175,4,/Users/piotrpijanowski/Documents/Studia/Semest...
7,mlp_adam_lr_1e_3,mlp,adam,0.0010,0.0,0.0,4,207128,0.656999,4,/Users/piotrpijanowski/Documents/Studia/Semest...
8,mlp_sgd_momentum_lr_1e_2,mlp,sgd,0.0100,0.9,0.0,4,207128,0.648076,4,/Users/piotrpijanowski/Documents/Studia/Semest...
9,mlp_adam_lr_3e_4,mlp,adam,0.0003,0.0,0.0,4,207128,0.595231,4,/Users/piotrpijanowski/Documents/Studia/Semest...
